Cleaning the data so that only men and women gender orientations are used and excludes others category in age group

In [3]:
import pandas as pd

# File path to your .ods file
file_path = "/Users/matthewng/Downloads/2023-Table-2-Selected-STI-diagnoses-and-rates-by-gender-sexual-orientation-age-group-and-ethnic-group.ods"

# Load the data starting from the actual header row (row 8 is 0-indexed)
warts_2i = pd.read_excel(file_path, sheet_name='2i_Warts', engine='odf', skiprows=8)

# Clean column names: remove leading/trailing spaces and replace non-breaking spaces
warts_2i.columns = warts_2i.columns.str.strip().str.replace('\xa0', ' ', regex=True)

# Filter for rows where Gender is Men or Women, and Age group is not "Other"
filtered_warts = warts_2i[
    (warts_2i['Gender and sexual orientation'].isin(['Men', 'Women'])) &
    (warts_2i['Age group'].str.strip().str.lower() != 'other')
].copy()

# Create 'country' and 'region' columns
filtered_warts['country'] = 'England'
filtered_warts['region'] = filtered_warts['Area of residence']

# Optional: reset index
filtered_warts.reset_index(drop=True, inplace=True)

# Display first few rows to verify
print(filtered_warts.head())


  Area of residence    STI Gender and sexual orientation Age group  \
0           England  Warts                           Men  13 to 14   
1           England  Warts                           Men  15 to 19   
2           England  Warts                           Men  20 to 24   
3           England  Warts                           Men  25 to 34   
4           England  Warts                           Men  35 to 44   

  2019 Numbers 2020 Numbers 2021 Numbers 2022 Numbers 2023 Numbers  \
0            2            4            1            1            2   
1          981          393          282          226          238   
2         9035         4538         3527         2637         2188   
3        11761         6624         7231         6391         6689   
4         4082         2209         2662         2611         2950   

   2019 Rates  2020 Rates  2021 Rates  2022 Rates  2023 Rates  country  \
0     0.30305    0.596543    0.144306    0.140592    0.281184  England   
1   61.335

Uploading code to MySQL database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# --- MySQL credentials ---
user = "root"
password = " "
host = "localhost"
database = "hpv_data"

# --- MySQL connection ---
engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

# --- Load and clean the data ---
file_path = "/Users/matthewng/Downloads/2023-Table-2-Selected-STI-diagnoses-and-rates-by-gender-sexual-orientation-age-group-and-ethnic-group.ods"
warts = pd.read_excel(file_path, sheet_name='2i_Warts', engine='odf', skiprows=8)
warts.columns = warts.columns.str.strip().str.replace('\xa0', ' ', regex=True)

# Filter the data
filtered = warts[
    (warts['Gender and sexual orientation'].isin(['Men', 'Women'])) &
    (warts['Age group'].str.strip().str.lower() != 'other')
].copy()

# Add country and region columns
filtered['country'] = 'England'
filtered['region'] = filtered['Area of residence']

# --- Transform into long format ---
melted = filtered.melt(
    id_vars=['country', 'region', 'STI', 'Gender and sexual orientation', 'Age group'],
    value_vars=[col for col in filtered.columns if 'Numbers' in col or 'Rates' in col],
    var_name='year_metric',
    value_name='value'
)

melted['year'] = melted['year_metric'].str.extract(r'(\d{4})')
melted['metric'] = melted['year_metric'].str.extract(r'(Numbers|Rates)')
melted.drop(columns='year_metric', inplace=True)

pivot = melted.pivot_table(
    index=['country', 'region', 'STI', 'Gender and sexual orientation', 'Age group', 'year'],
    columns='metric',
    values='value',
    aggfunc='first'
).reset_index()

pivot.columns.name = None
pivot.rename(columns={
    'STI': 'sti',
    'Gender and sexual orientation': 'gender_orientation',
    'Age group': 'age_group',
    'Numbers': 'diagnoses',
    'Rates': 'rate_per_100k'
}, inplace=True)

# --- Insert into lookup tables ---
def insert_unique(df, column, table_name, target_column='name'):
    unique_df = pd.DataFrame(df[column].unique(), columns=[target_column])
    unique_df.to_sql(table_name, con=engine, if_exists='append', index=False)

insert_unique(pivot, 'country', 'Country')

# Get country IDs
country_map = pd.read_sql('SELECT id, name FROM Country', con=engine).set_index('name')['id'].to_dict()
pivot['country_id'] = pivot['country'].map(country_map)

# Insert regions
region_df = pivot[['region', 'country_id']].drop_duplicates().rename(columns={'region': 'name'})
region_df.to_sql('Region', con=engine, if_exists='append', index=False)

# Get region IDs
region_map = pd.read_sql('SELECT id, name, country_id FROM Region', con=engine)
region_map['key'] = region_map['name'] + '|' + region_map['country_id'].astype(str)
pivot['region_key'] = pivot['region'] + '|' + pivot['country_id'].astype(str)
pivot = pivot.merge(region_map[['id', 'key']], left_on='region_key', right_on='key')
pivot.rename(columns={'id': 'region_id'}, inplace=True)

# Insert gender and age group
insert_unique(pivot, 'gender_orientation', 'GenderOrientation', 'label')
insert_unique(pivot, 'age_group', 'AgeGroup', 'group_label')

# Get IDs
gender_map = pd.read_sql('SELECT id, label FROM GenderOrientation', con=engine).set_index('label')['id'].to_dict()
age_map = pd.read_sql('SELECT id, group_label FROM AgeGroup', con=engine).set_index('group_label')['id'].to_dict()

pivot['gender_orientation_id'] = pivot['gender_orientation'].map(gender_map)
pivot['age_group_id'] = pivot['age_group'].map(age_map)

# --- Final upload to WartsStats ---
final = pivot[[
    'region_id', 'gender_orientation_id', 'age_group_id',
    'sti', 'year', 'diagnoses', 'rate_per_100k'
]]

final.to_sql('WartsStats', con=engine, if_exists='append', index=False)

print("✅ Upload complete.")


✅ Upload complete.
